# Skenario 2 — Cross-Language Information Retrieval (CLIR)
## Dense Retrieval dengan DistilBERT Multilingual (Baseline vs Fine-Tuned)

**Alur kerja notebook ini:**
1. Instalasi dependensi & setup environment
2. Import library dan inisialisasi PyTerrier
3. Definisi kelas `FaissRetriever` untuk dense retrieval
4. Load & preprocessing data (dokumen, kueri, qrels)
5. Eksperimen evaluasi: Baseline vs Fine-Tuned
6. Analisis hasil per-kueri dan penyimpanan laporan


## 1. Instalasi Dependensi & Setup Environment

Menginstall semua library yang dibutuhkan, meng-clone repository kode, dan menginisialisasi PyTerrier.
- **faiss-cpu** digunakan sebagai fallback aman karena `faiss-gpu` sering tidak tersedia di Kaggle.
- **sentence-transformers** untuk encoding teks menjadi vektor dense.
- **pyarabic** untuk normalisasi teks Arab.


In [1]:
import os
import sys
import subprocess
import torch

# --- 1a. Cek Ketersediaan GPU ---
if torch.cuda.is_available():
    print(f"✅ GPU Terdeteksi: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU tidak terdeteksi. Encoding akan berjalan di CPU (lebih lambat).")

# --- 1b. Install Library ---
print("\nMenginstall dependensi...")
libs = ["sentence-transformers", "python-terrier", "pyarabic", "faiss-cpu", "pandas", "tashaphyne"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + libs)
print("✅ Instalasi selesai.")

# --- 1c. Clone Repository ---
REPO_PATH = './skripsi-clir-code'
SRC_PATH  = os.path.join(REPO_PATH, 'src')

if os.path.exists(REPO_PATH):
    print("Repository sudah ada. Melakukan pull untuk update terbaru...")
    os.system(f"cd {REPO_PATH} && git pull")
else:
    print("Cloning repository...")
    os.system("git clone https://github.com/syifaurrr/skripsi-clir-code.git")

# Tambahkan folder src ke sys.path agar modul custom bisa di-import
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)
print(f"✅ Path '{SRC_PATH}' ditambahkan ke sys.path.")


✅ GPU Terdeteksi: Tesla T4

Menginstall dependensi...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.5/251.5 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.1 MB/s eta 0:00:00
✅ Instalasi selesai.
Cloning repository...


Cloning into 'skripsi-clir-code'...


✅ Path './skripsi-clir-code/src' ditambahkan ke sys.path.


## 2. Import Library & Inisialisasi PyTerrier

Mengimpor semua modul yang dibutuhkan, termasuk modul preprocessing Arab dari repository.
PyTerrier diinisialisasi sekali saja di awal karena memerlukan JVM (Java Virtual Machine).


In [2]:
import re
import numpy as np
import pandas as pd
import faiss
import pyterrier as pt
import pyarabic.araby as araby
from sentence_transformers import SentenceTransformer
from IPython.display import display

# Import modul preprocessing dari repository
try:
    from arabic_preprocessing import preprocess_pipeline
    print("✅ Modul arabic_preprocessing berhasil di-import.")
except ImportError as e:
    print(f"⚠️ Modul arabic_preprocessing tidak ditemukan: {e}")
    print("Pastikan struktur folder di repository benar: skripsi-clir-code/src/arabic_preprocessing.py")

# Inisialisasi PyTerrier (hanya dilakukan sekali)
if not pt.started():
    print("Menginisialisasi PyTerrier...")
    pt.init()
    print("✅ PyTerrier siap.")


/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:274: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN = re.compile(u"([^\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:276: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN_SPLIT = re.compile(u"([\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:281: SyntaxWarning: invalid escape sequence '\s'
  ARABIC_STRING = re.compile(u"([^\u0600-\u0652%s%s%s\s\d])" \
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:1237: SyntaxWarning: invalid escape sequence '\s'
  u"(?<=\s(%s|%s))%s" % (WAW, YEH, FATHA), \
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:1450: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(u"(?<=[\s\d])([%s])+"%(TASHKEEL_STRING),"",text,  re.UNICODE)


✅ Modul arabic_preprocessing berhasil di-import.
Menginisialisasi PyTerrier...
terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


/tmp/ipykernel_55/1136445848.py:19: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependencies.jar: 100%|██████████| 99.6M/99.6M [00:00<00:00, 173MB/s] 


Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar: 100%|██████████| 36.6k/36.6k [00:00<00:00, 31.3MB/s]

Done


✅ PyTerrier siap.


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_55/1136445848.py:21: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


## 3. Definisi Kelas `FaissRetriever`

Kelas ini membungkus FAISS sebagai `pt.Transformer` agar kompatibel dengan pipeline PyTerrier.

**Cara kerja:**
- Saat inisialisasi, dokumen di-encode menjadi vektor dense menggunakan SentenceTransformer.
- Vektor dokumen disimpan dalam FAISS `IndexFlatIP` (Inner Product ≈ Cosine Similarity setelah normalisasi L2).
- Saat `transform()` dipanggil, vektor kueri di-encode lalu dicari Top-1000 dokumen terdekat di index.

**Catatan patch DistilBERT:**  
DistilBERT tidak memiliki `token_type_ids`, namun tokenizer-nya kadang mendeklarasikannya. Baris patch menghapus field tersebut dari `model_input_names` untuk mencegah error.


In [3]:
class FaissRetriever(pt.Transformer):
    """
    Dense retriever berbasis FAISS + SentenceTransformer.
    Kompatibel dengan pipeline pt.Experiment (PyTerrier).
    """
    def __init__(self, model_name, doc_df=None, batch_size=32):
        self.model      = SentenceTransformer(model_name)
        self.index      = None
        self.doc_map    = {}
        self.batch_size = batch_size

        # Patch DistilBERT: hapus 'token_type_ids' yang tidak didukung
        tokenizer = getattr(self.model, 'tokenizer', None)
        if tokenizer and 'token_type_ids' in tokenizer.model_input_names:
            tokenizer.model_input_names.remove('token_type_ids')

        # Langsung buat index jika dokumen diberikan saat konstruksi
        if doc_df is not None:
            self._index_docs(doc_df)

    def _index_docs(self, doc_df):
        """Encode semua dokumen dan simpan ke index FAISS."""
        print(f"Encoding {len(doc_df)} dokumen (mungkin memakan waktu beberapa menit)...")
        docs    = doc_df['text_clean'].tolist()
        docnos  = doc_df['docno'].tolist()

        embeddings = self.model.encode(
            docs, batch_size=self.batch_size,
            show_progress_bar=True, convert_to_numpy=True
        )
        faiss.normalize_L2(embeddings)  # Wajib untuk Cosine Similarity via Inner Product

        dim        = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)

        # Peta index FAISS (integer) -> docno asli (string)
        self.doc_map = {i: docno for i, docno in enumerate(docnos)}
        print("✅ Indexing selesai.")

    def transform(self, queries):
        """Encode kueri dan kembalikan Top-1000 hasil retrieval."""
        q_texts      = queries['query'].tolist()
        q_embeddings = self.model.encode(
            q_texts, batch_size=self.batch_size,
            show_progress_bar=False, convert_to_numpy=True
        )
        faiss.normalize_L2(q_embeddings)

        k = 1000
        scores, ids = self.index.search(q_embeddings, k)

        results = []
        for i, qid in enumerate(queries['qid']):
            for j in range(k):
                idx = ids[i][j]
                if idx != -1:
                    results.append({
                        'qid'   : str(qid),
                        'docno' : self.doc_map[idx],
                        'score' : scores[i][j],
                        'rank'  : j + 1
                    })
        return pd.DataFrame(results)


## 4. Load & Preprocessing Data

Memuat tiga berkas data utama:
- **Dokumen** (`fathul_muin.csv`): Korpus teks berbahasa Arab yang akan diindeks.
- **Kueri** (`queries_indo.csv`): Pertanyaan dalam Bahasa Indonesia.
- **Qrels** (`qrels.csv`): Pasangan relevansi kueri-dokumen sebagai ground truth evaluasi.

Preprocessing dokumen Arab meliputi normalisasi karakter (harakat, tatweel, alef, ya/waw).
ID di semua dataframe distandarkan ke format string bersih untuk menghindari mismatch saat evaluasi.


In [4]:
# --- 4a. Fungsi Normalisasi Teks Arab ---
def normalize_arabic(text):
    """Membersihkan dan menormalisasi teks Arab."""
    if not isinstance(text, str):
        return ""
    text = araby.strip_tashkeel(text)   # Hapus harakat/tanda baca
    text = araby.strip_tatweel(text)    # Hapus tatweel (pemanjang)
    text = araby.normalize_alef(text)   # Normalisasi variasi huruf Alef
    text = re.sub(r'[ى]',  'ي', text)  # Alef maqsurah -> Ya
    text = re.sub(r'[ؤ]',  'و', text)  # Waw hamza -> Waw
    text = re.sub(r'[ئ]',  'ي', text)  # Ya hamza  -> Ya
    return text

# --- 4b. Konfigurasi Path Data ---
DATA_PATH       = './skripsi-clir-code/data'
raw_docs_path   = os.path.join(DATA_PATH, 'raw/fathul_muin.csv')
queries_path    = os.path.join(DATA_PATH, 'queries/queries_indo.csv')
qrels_path      = os.path.join(DATA_PATH, 'queries/qrels.csv')

# --- 4c. Load Dokumen ---
df_docs = pd.read_csv(raw_docs_path)
text_col = 'text' if 'text' in df_docs.columns else 'text_arabic'
df_docs['docno']      = df_docs['docno'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
df_docs['text_clean'] = df_docs[text_col].astype(str).apply(normalize_arabic)

# --- 4d. Load Kueri ---
df_queries = pd.read_csv(queries_path)
df_queries['qid'] = df_queries['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

# --- 4e. Load Qrels ---
qrels = pd.read_csv(qrels_path)
qrels['qid']   = qrels['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
qrels['docno'] = qrels['docno'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
qrels['label'] = qrels['label'].astype(int)

# --- Verifikasi format ID ---
print("Verifikasi format ID:")
print(f"  Sampel qid (kueri)  : {df_queries['qid'].iloc[0]!r}")
print(f"  Sampel qid (qrels)  : {qrels['qid'].iloc[0]!r}")
print(f"  Sampel docno (docs) : {df_docs['docno'].iloc[0]!r}")
print(f"  Sampel docno (qrels): {qrels['docno'].iloc[0]!r}")
print(f"\nJumlah dokumen : {len(df_docs)}")
print(f"Jumlah kueri   : {len(df_queries)}")
print(f"Jumlah qrels   : {len(qrels)}")


Verifikasi format ID:
  Sampel qid (kueri)  : '1'
  Sampel qid (qrels)  : '1'
  Sampel docno (docs) : 'Page_V01P031'
  Sampel docno (qrels): 'Page_V01P038'

Jumlah dokumen : 639
Jumlah kueri   : 153
Jumlah qrels   : 153


## 5. Eksperimen Evaluasi: Baseline vs Fine-Tuned

Membandingkan dua model:
1. **Baseline** — `distilbert-base-multilingual-cased` tanpa fine-tuning.
2. **Fine-Tuned** — Model yang sudah dilatih ulang dengan dataset domain (JH-POLO).

Eksperimen dijalankan dengan `perquery=True` agar dapat menganalisis performa per kueri.
Metrik yang digunakan:
- **Reciprocal Rank (RR)** — Kebalikan dari peringkat dokumen relevan pertama.
- **Success@K** (K = 10, 20, 50, 100) — Apakah dokumen relevan ditemukan dalam Top-K?


In [5]:
EVAL_METRICS = ["recip_rank", "success_10", "success_20", "success_50", "success_100"]

# --- 5a. Inisialisasi Model Baseline ---
print("Memuat Model Baseline (tanpa fine-tuning)...")
BASE_MODEL_NAME = 'distilbert-base-multilingual-cased'
retriever_base  = FaissRetriever(
    model_name=BASE_MODEL_NAME,
    doc_df=df_docs[['docno', 'text_clean']],
    batch_size=32
)

# --- 5b. Inisialisasi Model Fine-Tuned ---
# ⚠️ Sesuaikan path ini dengan lokasi model fine-tuned Anda di Kaggle Datasets
print("\nMemuat Model Fine-Tuned (JH-POLO)...")
FINETUNED_MODEL_PATH = '/kaggle/input/datasets/syifaur/distilbert-base-bs16-ep6-lr5e-05-wr0-1-seq128'
# norm
# FINETUNED_MODEL_PATH = '/kaggle/input/datasets/syifaur/distilbert-bs16-ep6-lr5e-05-wr0-1-seq128-norm'
retriever_finetuned  = FaissRetriever(
    model_name=FINETUNED_MODEL_PATH,
    doc_df=df_docs[['docno', 'text_clean']],
    batch_size=32
)

# --- 5c. Jalankan Eksperimen PyTerrier ---
print("\nMenjalankan pt.Experiment (perquery=True)...")
hasil_eval_perq = pt.Experiment(
    [retriever_base, retriever_finetuned],
    df_queries,
    qrels,
    eval_metrics=EVAL_METRICS,
    names=["Baseline (DistilBERT Base)", "DistilBERT Base Fine-Tuned (JH-POLO)"],
    validate="ignore",
    perquery=True
)

# --- 5d. Reshape: format panjang -> format lebar (metrik jadi kolom) ---
hasil_pivot = hasil_eval_perq.pivot_table(
    index=['name', 'qid'],
    columns='measure',
    values='value'
).reset_index()

# Merge metadata kueri (tipe & teks query) untuk analisis lebih lanjut
hasil_pivot['qid']   = hasil_pivot['qid'].astype(str)
df_queries['qid']    = df_queries['qid'].astype(str)

hasil_dengan_tipe = pd.merge(
    hasil_pivot,
    df_queries[['qid', 'query_type', 'query']],
    on='qid', how='left'
)
print("✅ Eksperimen selesai.")


No sentence-transformers model found with name distilbert-base-multilingual-cased. Creating a new one with mean pooling.


Memuat Model Baseline (tanpa fine-tuning)...


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

✅ Indexing selesai.

Memuat Model Fine-Tuned (JH-POLO)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Encoding 639 dokumen (mungkin memakan waktu beberapa menit)...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

✅ Indexing selesai.

Menjalankan pt.Experiment (perquery=True)...
✅ Eksperimen selesai.


## 6. Analisis & Penyimpanan Hasil

Hasil evaluasi ditampilkan dalam tiga level:
1. **Overall** — Rata-rata metrik seluruh kueri untuk tiap model.
2. **Per tipe kueri** — Rata-rata metrik dikelompokkan berdasarkan kategori kueri.
3. **Detail per kueri** — Informasi rank dokumen relevan ditemukan dan label Hit/Miss@K untuk tiap kueri.

Semua tabel disimpan sebagai CSV untuk keperluan penulisan skripsi.


In [6]:
# Helper: konversi kolom success ke persen dan bulatkan recip_rank
def format_metrics(df, groupby_cols):
    result = df.groupby(groupby_cols)[EVAL_METRICS].mean().reset_index()
    for k in [10, 20, 50, 100]:
        result[f'success_{k} (%)'] = (result[f'success_{k}'] * 100).round(2)
    result['recip_rank'] = result['recip_rank'].round(4)
    return result

DISPLAY_COLS_BASE = ['name', 'recip_rank', 'success_10 (%)', 'success_20 (%)', 'success_50 (%)', 'success_100 (%)']

# --- 6a. Hasil Keseluruhan (Overall) ---
print("=" * 60)
print("🎯 HASIL KESELURUHAN (OVERALL)")
print("=" * 60)
overall_metrics = format_metrics(hasil_dengan_tipe, ['name'])
display(overall_metrics[DISPLAY_COLS_BASE])

# --- 6b. Hasil Per Tipe Kueri ---
print("\n" + "=" * 60)
print("📊 HASIL BERDASARKAN TIPE KUERI")
print("=" * 60)
per_type_metrics = format_metrics(hasil_dengan_tipe, ['name', 'query_type'])
display(per_type_metrics[['name', 'query_type'] + DISPLAY_COLS_BASE[1:]])

# --- 6c. Analisis Detail Per Kueri (Hit/Miss@K) ---
print("\n" + "=" * 60)
print("🔍 ANALISIS DETAIL PER KUERI")
print("=" * 60)

detail_kueri = hasil_dengan_tipe.copy()

# Konversi Reciprocal Rank -> estimasi peringkat dokumen relevan
detail_kueri['rank_ditemukan'] = detail_kueri['recip_rank'].apply(
    lambda x: int(round(1 / x)) if x > 0 else "Tidak Ketemu"
)
# Label visual Hit/Miss untuk tiap threshold K
for k in [10, 20, 50, 100]:
    detail_kueri[f'Hit@{k}'] = detail_kueri[f'success_{k}'].apply(
        lambda x: '✅ Hit' if x == 1.0 else '❌ Miss'
    )

# Urutkan: Model -> Tipe Kueri -> Nomor Kueri (numerik)
detail_kueri['qid_int'] = pd.to_numeric(detail_kueri['qid'], errors='coerce').fillna(0).astype(int)
detail_kueri = detail_kueri.sort_values(by=['name', 'query_type', 'qid_int']).drop(columns='qid_int')

KOLOM_TAMPIL = ['name', 'qid', 'query_type', 'query', 'rank_ditemukan',
                'Hit@10', 'Hit@20', 'Hit@50', 'Hit@100']
display(detail_kueri[KOLOM_TAMPIL].head(20))

# --- 6d. Simpan Semua Laporan ke CSV ---
detail_kueri[KOLOM_TAMPIL].to_csv('skenario2_detail_per_kueri.csv',        index=False)
per_type_metrics.to_csv(           'skenario2_evaluasi_per_tipe_kueri.csv', index=False)
overall_metrics.to_csv(            'skenario2_evaluasi_overall.csv',        index=False)

print("\n📑 Seluruh laporan berhasil disimpan:")
print("  - skenario2_detail_per_kueri.csv")
print("  - skenario2_evaluasi_per_tipe_kueri.csv")
print("  - skenario2_evaluasi_overall.csv")


🎯 HASIL KESELURUHAN (OVERALL)


,name,recip_rank,success_10 (%),success_20 (%),success_50 (%),success_100 (%)
0,Baseline (DistilBERT Base),0.0149,1.96,5.88,13.73,31.37
1,DistilBERT Base Fine-Tuned (JH-POLO),0.2007,35.29,45.10,55.56,65.36



📊 HASIL BERDASARKAN TIPE KUERI


,name,query_type,recip_rank,success_10 (%),success_20 (%),success_50 (%),success_100 (%)
0,Baseline (DistilBERT Base),1,0.0154,1.96,9.80,13.73,33.33
1,Baseline (DistilBERT Base),2,0.0106,0.00,3.92,17.65,33.33
2,Baseline (DistilBERT Base),3,0.0187,3.92,3.92,9.80,27.45
3,DistilBERT Base Fine-Tuned (JH-POLO),1,0.2345,37.25,45.10,50.98,62.75
4,DistilBERT Base Fine-Tuned (JH-POLO),2,0.1659,33.33,45.10,60.78,66.67
5,DistilBERT Base Fine-Tuned (JH-POLO),3,0.2018,35.29,45.10,54.90,66.67



🔍 ANALISIS DETAIL PER KUERI


,name,qid,query_type,query,rank_ditemukan,Hit@10,Hit@20,Hit@50,Hit@100
0,Baseline (DistilBERT Base),1,1,Kewajiban orang tua untuk memerintahkan sholat...,29,❌ Miss,❌ Miss,✅ Hit,✅ Hit
87,Baseline (DistilBERT Base),4,1,Kesunnahan membaca basmalah dalam wudhu,14,❌ Miss,✅ Hit,✅ Hit,✅ Hit
120,Baseline (DistilBERT Base),7,1,Alasan-alasan yang membatalkan wudhu,395,❌ Miss,❌ Miss,❌ Miss,❌ Miss
1,Baseline (DistilBERT Base),10,1,Syarat sah sholat yang kedua,266,❌ Miss,❌ Miss,❌ Miss,❌ Miss
34,Baseline (DistilBERT Base),13,1,Tata cara mensucikan najis (air seni) yang tel...,435,❌ Miss,❌ Miss,❌ Miss,❌ Miss
61,Baseline (DistilBERT Base),16,1,Kewajiban mengingatkan seseorang yang pakaiann...,83,❌ Miss,❌ Miss,❌ Miss,✅ Hit
64,Baseline (DistilBERT Base),19,1,Kesunahan membaca doa qunut di dalam sholat subuh,19,❌ Miss,✅ Hit,✅ Hit,✅ Hit
68,Baseline (DistilBERT Base),22,1,Solusi bagi makmum ketika lupa membaca surah a...,85,❌ Miss,❌ Miss,❌ Miss,✅ Hit
71,Baseline (DistilBERT Base),25,1,Hukum melakukan sunnah ab'ad yang terlupakan s...,207,❌ Miss,❌ Miss,❌ Miss,❌ Miss
74,Baseline (DistilBERT Base),28,1,Kesunnahan adzan pada selain sholat fardhu,363,❌ Miss,❌ Miss,❌ Miss,❌ Miss



📑 Seluruh laporan berhasil disimpan:
  - skenario2_detail_per_kueri.csv
  - skenario2_evaluasi_per_tipe_kueri.csv
  - skenario2_evaluasi_overall.csv
